In [ ]:
"""
prepare_NIH_for_test.py
========================
يحوّل NIH Chest X-rays dataset إلى هيكل ImageFolder جاهز للاختبار:

    NIH_test_ready/
        NORMAL/       ← صور "No Finding"
        PNEUMONIA/    ← صور "Pneumonia"

المدخلات المطلوبة:
    - Data_Entry_2017.csv  (موجود داخل الـ zip مع الصور)
    - مجلد الصور (images_001 .. images_012 أو images/ مباشرة)

الاستخدام:
    python prepare_NIH_for_test.py
    ثم عدّلي NIH_ROOT في السطر التالي لمسار الـ dataset عندكِ
"""

import os
import shutil
import pandas as pd
from pathlib import Path

# ══════════════════════════════════════════════════════
#  ← غيّري المسارات دي بس
# ══════════════════════════════════════════════════════
NIH_ROOT   = r"E:\RashaHagagMaster\Pneumonia\NIH Chest X-rays"
CSV_PATH   = os.path.join(NIH_ROOT, "Data_Entry_2017.csv")
# هنا هنخلي IMAGES_DIR يشاور على الفولدر الرئيسي اللي جواه المجلدات الفرعية
IMAGES_DIR = NIH_ROOT
OUTPUT_DIR = os.path.join(NIH_ROOT, "NIH_test_ready")

# عدد الصور لكل كلاس (None = كل الصور المتاحة)
MAX_NORMAL    = 2000   # نأخذ sample متوازن — عندنا ~60k normal
MAX_PNEUMONIA = None   # نأخذ كل صور الـ Pneumonia (~1,431 صورة)
# ══════════════════════════════════════════════════════

def find_image(filename: str, images_dir: str) -> str | None:
    """يبحث عن الصورة داخل مجلدات images_001 وصولاً إلى images_012"""
    # بنجرب ندور في كل المجلدات اللي بتبدأ بكلمة images_
    for i in range(1, 13):
        sub_folder = f"images_{i:03d}"  # دي هتطلع images_001, images_002...
        full_path = os.path.join(images_dir, sub_folder, "images", filename)
        
        # لو المسار عندك مفيهوش كلمة images تانية جوه المجلد الفرعي، جربي ده:
        if not os.path.exists(full_path):
            full_path = os.path.join(images_dir, sub_folder, filename)
            
        if os.path.exists(full_path):
            return full_path
    return None


def main():
    print("=" * 55)
    print("NIH Chest X-rays → ImageFolder Preparer")
    print("=" * 55)

    # ── 1. قراءة الـ CSV ──────────────────────────────────
    print(f"\n[1/4] قراءة {CSV_PATH} ...")
    df = pd.read_csv(CSV_PATH)
    print(f"      إجمالي الصور في الـ CSV: {len(df):,}")

    # ── 2. فلترة الكلاسين المطلوبين ──────────────────────
    print("\n[2/4] فلترة الكلاسين ...")

    # No Finding: صور بدون أي مرض — كلاسنا NORMAL
    # Pneumonia:  صور Pneumonia فقط (بدون أمراض أخرى مصاحبة)
    df_normal    = df[df["Finding Labels"] == "No Finding"].copy()
    df_pneumonia = df[df["Finding Labels"].str.contains("Pneumonia", case=False)].copy()

    print(f"      No Finding  (NORMAL)   : {len(df_normal):,} صورة")
    print(f"      Pneumonia              : {len(df_pneumonia):,} صورة")

    # Sample متوازن
    if MAX_NORMAL and len(df_normal) > MAX_NORMAL:
        df_normal = df_normal.sample(MAX_NORMAL, random_state=42)
        print(f"      تم أخذ sample من NORMAL  : {MAX_NORMAL:,} صورة")

    if MAX_PNEUMONIA and len(df_pneumonia) > MAX_PNEUMONIA:
        df_pneumonia = df_pneumonia.sample(MAX_PNEUMONIA, random_state=42)

    # ── 3. إنشاء الـ folders ──────────────────────────────
    print("\n[3/4] إنشاء مجلدات الإخراج ...")
    normal_dir    = os.path.join(OUTPUT_DIR, "NORMAL")
    pneumonia_dir = os.path.join(OUTPUT_DIR, "PNEUMONIA")
    os.makedirs(normal_dir,    exist_ok=True)
    os.makedirs(pneumonia_dir, exist_ok=True)
    print(f"      {normal_dir}")
    print(f"      {pneumonia_dir}")

    # ── 4. نسخ الصور ──────────────────────────────────────
    print("\n[4/4] نسخ الصور ...")

    def copy_images(df_subset, dest_dir, label):
        copied, missing = 0, 0
        for fname in df_subset["Image Index"]:
            src = find_image(fname, IMAGES_DIR)
            if src:
                shutil.copy2(src, os.path.join(dest_dir, fname))
                copied += 1
            else:
                missing += 1
        print(f"      {label:10s}: {copied:,} منسوخة  |  {missing:,} غير موجودة")
        return copied, missing

    n_copied, n_miss = copy_images(df_normal,    normal_dir,    "NORMAL")
    p_copied, p_miss = copy_images(df_pneumonia, pneumonia_dir, "PNEUMONIA")

    # ── ملخص ──────────────────────────────────────────────
    print("\n" + "=" * 55)
    print("اكتمل التجهيز")
    print("=" * 55)
    print(f"  NORMAL    : {n_copied:,} صورة  {'⚠ ' + str(n_miss) + ' غير موجودة' if n_miss else ''}")
    print(f"  PNEUMONIA : {p_copied:,} صورة  {'⚠ ' + str(p_miss) + ' غير موجودة' if p_miss else ''}")
    print(f"\n  المجلد الجاهز: {OUTPUT_DIR}")
    print("\n  استخدمي هذا المسار في testNewdata_02.ipynb:")
    print(f"  new_data_path = r'{OUTPUT_DIR}'")
    print("=" * 55)

    # ── تحقق سريع من class_to_idx المتوقع ────────────────
    print("\n  class_to_idx المتوقع عند تشغيل الاختبار:")
    import json
    classes = sorted(os.listdir(OUTPUT_DIR))
    mapping = {c: i for i, c in enumerate(classes)}
    print(" ", json.dumps(mapping, ensure_ascii=False))
    assert mapping.get("NORMAL") == 0,    "خطأ: ترتيب NORMAL غير صحيح"
    assert mapping.get("PNEUMONIA") == 1, "خطأ: ترتيب PNEUMONIA غير صحيح"
    print("  class_to_idx صحيح ✓")


if __name__ == "__main__":
    main()


NIH Chest X-rays → ImageFolder Preparer

[1/4] قراءة E:\RashaHagagMaster\Pneumonia\NIH Chest X-rays\Data_Entry_2017.csv ...
      إجمالي الصور في الـ CSV: 112,120

[2/4] فلترة الكلاسين ...
      No Finding  (NORMAL)   : 60,361 صورة
      Pneumonia              : 1,431 صورة
      تم أخذ sample من NORMAL  : 2,000 صورة

[3/4] إنشاء مجلدات الإخراج ...
      E:\RashaHagagMaster\Pneumonia\NIH Chest X-rays\NIH_test_ready\NORMAL
      E:\RashaHagagMaster\Pneumonia\NIH Chest X-rays\NIH_test_ready\PNEUMONIA

[4/4] نسخ الصور ...
      NORMAL    : 0 منسوخة  |  2,000 غير موجودة
      PNEUMONIA : 0 منسوخة  |  1,431 غير موجودة

اكتمل التجهيز
  NORMAL    : 0 صورة  ⚠ 2000 غير موجودة
  PNEUMONIA : 0 صورة  ⚠ 1431 غير موجودة

  المجلد الجاهز: E:\RashaHagagMaster\Pneumonia\NIH Chest X-rays\NIH_test_ready

  استخدمي هذا المسار في testNewdata_02.ipynb:
  new_data_path = r'E:\RashaHagagMaster\Pneumonia\NIH Chest X-rays\NIH_test_ready'

  class_to_idx المتوقع عند تشغيل الاختبار:
  {"NORMAL": 0, "PNEUMONIA": 1}